# Toxion: Data check

Looks at the cleaned Tox21 table: shape, class balance per endpoint, missing labels, and a few structures.

**Local:** `uv sync --extra notebook`, then `uv run jupyter lab` and open this file.
**Colab:** upload or clone the repo, run `%pip install -e /content/toxion` in a cell, then run the cells below. Run this from the repo folder.

In [ ]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw/tox21.csv.gz"
CLEAN = ROOT / "data/processed/tox21_clean.parquet"
print("repo root:", ROOT)

In [ ]:
from toxion.data.loader import load_and_clean, load_clean, save_clean

if not CLEAN.exists():
    if not RAW.exists():
        raise FileNotFoundError(f"{RAW} not found. Run: python scripts/download_data.py")
    save_clean(load_and_clean(str(RAW)), str(CLEAN))

df = load_clean(str(CLEAN))
print(df.shape)
df.head()

## Overall label coverage
NaN means 'not tested', not 'inactive'.

In [ ]:
from toxion.data.summary import overall_stats, summarize_labels

overall_stats(df)

## Class balance per endpoint

In [ ]:
summary = summarize_labels(df)
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary["positive_rate"].plot.bar(ax=axes[0], title="Positive rate per endpoint")
summary["labeled"].plot.bar(ax=axes[1], title="Labeled molecules per endpoint")
for ax in axes:
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## Labels per molecule

In [ ]:
from toxion.data.loader import TOX21_TASKS

df[TOX21_TASKS].notna().sum(axis=1).value_counts().sort_index().plot.bar(
    title="Number of endpoints with a label, per molecule"
)
plt.xlabel("labeled endpoints")
plt.ylabel("molecules")
plt.show()

## A random sample of structures

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

sample = df.sample(min(12, len(df)), random_state=0)
Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in sample["smiles_canonical"]],
    molsPerRow=4,
    subImgSize=(240, 240),
    legends=[s[:30] for s in sample["smiles_canonical"]],
)